# استراتژی‌های اختیار معامله

In [ ]:
# =====================================================================
# بخش 1: واردات کتابخانه‌ها و تنظیمات اولیه
# =====================================================================

import sys
import os
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from scipy.stats import norm

# تنظیم مسیر پروژه
notebook_path = os.path.abspath('')
root_dir = Path(notebook_path).parent
sys.path.append(str(root_dir))

# واردات ماژول‌های پروژه
from data.downloader import MarketDownloader
from data.cleaner import DataCleaner
from config import (
    EXERCISE_TAX_RATE,
    get_symbol_market,
    get_symbol_kind,
    get_commission_rate,
    get_exercise_fee_rate,
)

In [ ]:
# =====================================================================
# بخش 2: بارگذاری و پاکسازی داده‌ها
# =====================================================================

def load_and_clean_data():
    """
    بارگذاری داده‌ها از TSE و انجام پاکسازی اولیه
    """
    df_raw = MarketDownloader.from_tsetmc_direct()
    df_cleaned = DataCleaner.clean(df_raw)
    df_final = DataCleaner.add_derived_columns(df_cleaned)
    return df_final

def filter_options(df, exclude_underlying=['اهرم'], exclude_name_pattern=['1405/04', '1405-04']):
    """
    فیلتر کردن گزینه‌های اختیار خرید با حذف موارد نامطلوب
    """
    filtered = df[
        (df['DaysToMaturity'] > 2.0) &
        (df['Type'].apply(lambda x: x.name == 'CALL'))
    ].copy()
    
    exclude_mask = (
        (filtered['UnderlyingTicker'].isin(exclude_underlying)) & 
        (filtered['Name'].str.contains('|'.join(exclude_name_pattern), na=False))
    )
    filtered = filtered[~exclude_mask].copy()
    return filtered

# بارگذاری داده‌ها
df_raw = load_and_clean_data()

# فیلتر کردن گزینه‌های اختیار خرید
filtered_options = filter_options(df_raw)

# نمایش نمونه داده
filtered_options.head(3)

In [ ]:
# =====================================================================
# بخش 3: توابع محاسباتی استراتژی‌ها
# =====================================================================

def covered_call_with_fees(ticker, premium_call, stock_price, strike_price, contract_size,
                           opt_sell_commission, stock_buy_commission, exercise_fee_rate, 
                           exercise_tax_rate, days):
    """
    محاسبه بازده استراتژی Covered Call با احتساب کارمزدها
    """
    
    # ====================== 1. کارمزدهای ورود ======================
    option_fee = -round(premium_call * contract_size * opt_sell_commission, 0)
    stock_buy_fee = -round(stock_price * contract_size * stock_buy_commission, 0)
    entry_fees = option_fee + stock_buy_fee

    # ====================== 2. کارمزدهای خروج/اعمال ======================
    exercise_fee = -round(strike_price * contract_size * exercise_fee_rate, 0)
    exercise_tax = -round(strike_price * contract_size * exercise_tax_rate, 0)

    # ====================== 3. مبالغ اصلی ======================
    premium_received = premium_call * contract_size
    stock_cost = -stock_price * contract_size

    # ====================== 4. سرمایه اولیه خالص ======================
    net_investment = stock_cost + premium_received + entry_fees

    # ====================== 5. دریافت وجه در سررسید ======================
    strike_received = strike_price * contract_size
    net_received = strike_received + exercise_fee + exercise_tax

    # ====================== 6. سود خالص ======================
    net_profit = net_received + net_investment

    # ====================== 7. درصد بازده ======================
    profit_percent = (round((net_profit / abs(net_investment)) * 100, 2) if net_investment != 0 else 0)
    monthly_return = round(profit_percent * (30 / days), 2)

    # ====================== 8. قیمت سربه‌سر ======================
    downside_protection = premium_received + entry_fees + exercise_fee + exercise_tax
    break_even_price = round(stock_price - (downside_protection / contract_size), 0)

    # ====================== 9. درصد افت مجاز ======================
    max_drop_percent = round(((stock_price - break_even_price) / stock_price) * 100, 2)

    return {
        'net_profit': net_profit,
        'profit_percent': profit_percent,
        'monthly_return': monthly_return,
        'break_even_price': break_even_price,
        'max_drop_percent': max_drop_percent
    }


def long_call_with_fees(ticker, premium_call, stock_price, strike_price, contract_size,
                        opt_buy_commission, exercise_fee_rate, days):
    """
    محاسبه بازده استراتژی Long Call با احتساب کارمزدها
    """

    # ========== 1. محاسبه هزینه‌های ورود ==========
    premium_total = -round(premium_call * contract_size, 0)
    entry_fee = round(premium_total * opt_buy_commission, 0)
    
    # ========== 2. سرمایه اولیه ==========
    initial_investment = premium_total + entry_fee
    
    # ========== 3. محاسبه سود ناخالص در سررسید ==========
    intrinsic_value = max(0, stock_price - strike_price) * contract_size
    
    # ========== 4. کارمزد اعمال ==========
    exercise_fee = 0
    if stock_price > strike_price:
        settlement_amount = strike_price * contract_size
        exercise_fee = -round(settlement_amount * exercise_fee_rate, 0)
        
    # ========== 5. سود خالص نهایی ==========
    net_profit = intrinsic_value + initial_investment + exercise_fee
    
    # ========== 6. بازده درصدی ==========
    profit_percent = round((net_profit / abs(initial_investment)) * 100, 2) if initial_investment != 0 else 0
    monthly_return = round(profit_percent * (30 / days), 2)
    
    # ========== 7. نقطه سربه‌سر ==========
    total_cost_per_share = premium_call + (abs(entry_fee) / contract_size) + (abs(exercise_fee) / contract_size) if stock_price > strike_price else premium_call
    break_even_price = round(strike_price + total_cost_per_share, 0)
    
    # ========== 8. درصد فاصله تا نقطه سربه‌سر ==========
    if break_even_price != 0:
        break_even_percent = round(((break_even_price - stock_price) / stock_price) * 100, 2)
    else:
        break_even_percent = 0
    
    return {
        'net_profit': net_profit,
        'profit_percent': profit_percent,
        'monthly_return': monthly_return,
        'break_even_price': break_even_price,
        'break_even_percent': break_even_percent,
        'intrinsic_value': intrinsic_value,
        'fees_total': entry_fee + exercise_fee
    }

In [ ]:
# =====================================================================
# بخش 4: توابع محاسبه دلتا و امتیازدهی (ویژه Covered Call)
# =====================================================================

def calculate_black_scholes_delta(S, K, T, r, sigma):
    """
    محاسبه دلتای اختیار خرید با فرمول بلک-شولز
    """
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.50
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    return round(float(np.clip(norm.cdf(d1), 0.0, 1.0)), 4)


def score_covered_call(row, hv_col='Volatility', delta_col='Delta'):
    """
    سیستم امتیازدهی استراتژی Covered Call
    """
    monthly_return = row['monthly_return_%']
    max_drop = row['max_drop_%']
    dte = row['days_to_maturity']
    stock_price = row['stock_price']
    strike = row['strike']
    hv = row[hv_col]
    delta = row[delta_col]

    return_score = np.clip(monthly_return / 12.0, 0.0, 1.0)
    protection_score = np.clip(max_drop / 25.0, 0.0, 1.0)

    expected_move = 1.5 * hv * np.sqrt(dte / 365.0) * 100
    downside_score = np.clip(max_drop / expected_move, 0.0, 1.0) if expected_move > 0 else 1.0

    net_delta = abs(1.0 - delta)
    delta_score = np.exp(-2.0 * net_delta)

    moneyness = (stock_price - strike) / stock_price if stock_price > 0 else 0
    itm_penalty = (1.0 - np.exp(-6.0 * moneyness)) if moneyness > 0 else 0.0

    final_score = (
        0.30 * return_score +
        0.25 * protection_score +
        0.25 * downside_score +
        0.10 * delta_score -
        0.10 * itm_penalty
    )
    return round(np.clip(final_score * 100, 0, 100), 2)


def load_volatility_data(vol_file="daily_market_volatility.xlsx"):
    """
    بارگذاری داده‌های نوسان از فایل
    """
    if os.path.exists(vol_file):
        df_vol = pd.read_excel(vol_file)
        return df_vol[['UnderlyingTicker', 'Volatility']].rename(columns={'UnderlyingTicker': 'underlying'})
    return pd.DataFrame(columns=['underlying', 'Volatility'])


def process_covered_call_scoring(df_input, df_volatility, r_free=0.25):
    """
    پردازش و امتیازدهی نتایج Covered Call
    """
    df_merged = pd.merge(df_input, df_volatility, on='underlying', how='left')
    df_merged['Volatility'] = df_merged['Volatility'].fillna(0.45)
    
    df_merged['Delta'] = df_merged.apply(
        lambda r: calculate_black_scholes_delta(
            S=r['stock_price'],
            K=r['strike'],
            T=r['days_to_maturity'] / 365.0,
            r=r_free,
            sigma=r['Volatility']
        ), axis=1
    )
    
    df_merged['score'] = df_merged.apply(score_covered_call, axis=1)
    return df_merged.sort_values(by='score', ascending=False).reset_index(drop=True)

In [ ]:
# =====================================================================
# بخش 5: اجرای استراتژی Covered Call
# =====================================================================

def run_covered_call_strategy(df_options):
    """
    اجرای استراتژی Covered Call روی داده‌های فیلتر شده
    """
    results = []
    
    for underlying_symbol, group in df_options.groupby('UnderlyingTicker'):
        market = get_symbol_market(underlying_symbol)
        kind = get_symbol_kind(underlying_symbol)

        opt_sell_commission = get_commission_rate(market, 'option', False)
        stock_buy_commission = get_commission_rate(market, kind, True)
        exercise_fee_rate = get_exercise_fee_rate(market, kind)
        exercise_tax_rate = EXERCISE_TAX_RATE

        for _, item in group.iterrows():
            ticker = item['Ticker']
            strike_price = item['StrikePrice']
            premium_call = item['BidPrice']
            stock_price = item['UnderlyingPrice']
            contract_size = item['ContractSize']
            days = item['DaysToMaturity']
            
            result = covered_call_with_fees(
                ticker, premium_call, stock_price, strike_price, contract_size,
                opt_sell_commission, stock_buy_commission,
                exercise_fee_rate, exercise_tax_rate, days
            )

            results.append({
                'underlying': underlying_symbol,
                'option_symbol': ticker,
                'strike': strike_price,
                'premium': round(premium_call, 0),
                'stock_price': round(stock_price, 0),
                'net_profit': result['net_profit'],
                'profit_percent': result['profit_percent'],
                'monthly_return_%': result['monthly_return'],
                'break_even_price': result['break_even_price'],
                'max_drop_%': result['max_drop_percent'],
                'status': getattr(item['OptionStatus'], 'value', item['OptionStatus']),
                'days_to_maturity': days,
                'volume': int(item.get('Volume', 0))
            })
    
    return pd.DataFrame(results)

covered_call_results = run_covered_call_strategy(filtered_options)

# بارگذاری داده‌های نوسان و امتیازدهی
vol_data = load_volatility_data()
scored_covered_call = process_covered_call_scoring(covered_call_results, vol_data)

# فیلتر نهایی با ضریب DTE
scored_covered_call['dte_factor'] = (scored_covered_call['days_to_maturity'] / 30) ** 0.5
scored_covered_call['dte_factor'] = scored_covered_call['dte_factor'].clip(lower=0.3, upper=2.5)
scored_covered_call['max_drop_threshold'] = 10.0 * scored_covered_call['dte_factor']

final_covered_call = scored_covered_call[
    scored_covered_call['max_drop_%'] >= scored_covered_call['max_drop_threshold']].copy()
final_covered_call = final_covered_call.sort_values(
    by='monthly_return_%', ascending=False).reset_index(drop=True)

In [ ]:
# =====================================================================
# بخش 6: اجرای استراتژی Long Call
# =====================================================================

def run_long_call_strategy(df_options):
    """
    اجرای استراتژی Long Call روی داده‌های فیلتر شده
    """
    results = []
    
    for underlying_symbol, group in df_options.groupby('UnderlyingTicker'):
        market = get_symbol_market(underlying_symbol)
        kind = get_symbol_kind(underlying_symbol)

        opt_buy_commission = get_commission_rate(market, 'option', True)
        exercise_fee_rate = get_exercise_fee_rate(market, kind)

        for _, item in group.iterrows():
            ticker = item['Ticker']
            strike_price = item['StrikePrice']
            premium_call = item['AskPrice']
            stock_price = item['UnderlyingPrice']
            contract_size = item['ContractSize']
            days = item['DaysToMaturity']
            
            result = long_call_with_fees(
                ticker, premium_call, stock_price, strike_price, contract_size,
                opt_buy_commission, exercise_fee_rate, days)

            results.append({
                'underlying': underlying_symbol,
                'option_symbol': ticker,
                'strike': strike_price,
                'premium': round(premium_call, 0),
                'stock_price': round(stock_price, 0),
                'net_profit': result['net_profit'],
                'profit_percent': result['profit_percent'],
                'monthly_return_%': result['monthly_return'],
                'break_even_price': result['break_even_price'],
                'break_even_percent': result['break_even_percent'],
                'days_to_maturity': days,
                'volume': int(item.get('Volume', 0))})
    
    return pd.DataFrame(results)

long_call_results = run_long_call_strategy(filtered_options)

# فیلتر بر اساس درصد فاصله تا نقطه سربه‌سر
long_call_filtered = long_call_results[
    long_call_results['break_even_percent'] <= 12].copy()
long_call_filtered = long_call_filtered.sort_values(
    ['break_even_percent', 'monthly_return_%'], 
    ascending=[True, True]).reset_index(drop=True)

In [ ]:
# =====================================================================
# بخش 7: ذخیره‌سازی نتایج در فایل اکسل
# =====================================================================
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

def save_results_to_excel(covered_call_df, long_call_df, filename_prefix="options_strategies"):
    """
    ذخیره نتایج هر دو استراتژی در یک فایل اکسل با شیت‌های جداگانه
    """
    # تنظیمات استایل
    header_font = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
    header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
    alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    body_font = Font(name='Segoe UI', size=10)
    gray_font = Font(color='808080', italic=True, name='Segoe UI', size=10)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{filename_prefix}_{timestamp}.xlsx"
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        # شیت Covered Call
        if not covered_call_df.empty:
            covered_call_df.to_excel(writer, sheet_name='Covered_Call', index=False)
            _apply_excel_styling(writer, 'Covered_Call', covered_call_df, 
                                 header_font, header_fill, alignment, body_font, gray_font)
        
        # شیت Long Call
        if not long_call_df.empty:
            long_call_df.to_excel(writer, sheet_name='Long_Call', index=False)
            _apply_excel_styling(writer, 'Long_Call', long_call_df,
                                 header_font, header_fill, alignment, body_font, gray_font)

    return filename


def _apply_excel_styling(writer, sheet_name, df, header_font, header_fill, alignment, body_font, gray_font):
    """اعمال استایل به شیت اکسل"""
    worksheet = writer.sheets[sheet_name]
    
    # استایل هدر
    for col_idx in range(1, len(df.columns) + 1):
        cell = worksheet.cell(row=1, column=col_idx)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = alignment

    # استایل بدنه
    columns_list = df.columns.tolist()
    for row_idx, row in enumerate(df.itertuples(index=False), start=2):
        for col_idx, col_name in enumerate(columns_list, start=1):
            cell = worksheet.cell(row=row_idx, column=col_idx)
            val = row[col_idx - 1]

            if val is None or pd.isna(val):
                cell.value = "-"
                cell.font = gray_font
            else:
                cell.font = body_font
            cell.alignment = alignment

    # تنظیم عرض ستون‌ها
    for col in worksheet.columns:
        max_len = 0
        for cell in col:
            val = str(cell.value or '')
            actual_len = sum(2 if ord(c) > 128 else 1 for c in val)
            if actual_len > max_len:
                max_len = actual_len
        col_letter = get_column_letter(col[0].column)
        worksheet.column_dimensions[col_letter].width = min((max_len + 4), 50)

    # فیلتر و Freeze Panes
    worksheet.auto_filter.ref = f"A1:{get_column_letter(len(df.columns))}{len(df) + 1}"
    worksheet.freeze_panes = 'A2'

# ذخیره نتایج
output_file = save_results_to_excel(
    final_covered_call, long_call_filtered, "options_strategies")